# Mini RAG: Ask Questions About AI Articles

A simple RAG system that:
- Loads markdown documents about AI from `knowledge_base/`
- Splits them into chunks
- Creates vector embeddings (HuggingFace)
- Stores them in ChromaDB
- Visualizes the embeddings with t-SNE
- Lets you ask questions about the documents

## 1. Imports and setup

In [1]:
import os
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go

load_dotenv(override=True)

# OpenRouter: GPT-4o mini
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# Paths
KNOWLEDGE_BASE = Path("knowledge_base")
CHROMA_PATH = "chroma_ai_articles"
COLLECTION_NAME = "ai_articles"

CHUNK_SIZE = 400
CHUNK_OVERLAP = 80

## 2. Load and chunk documents

In [2]:
def load_docs(base_path: Path):
    """Load all .md files under base_path. Returns list of (text, source)."""
    docs = []
    for f in sorted(base_path.rglob("*.md")):
        text = f.read_text(encoding="utf-8")
        docs.append((text, str(f)))
    return docs

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start = end - overlap
    return chunks

documents = load_docs(KNOWLEDGE_BASE)
all_chunks = []
for text, source in documents:
    for i, c in enumerate(chunk_text(text)):
        doc_name = Path(source).stem
        all_chunks.append({"content": c, "source": source, "doc": doc_name})

print(f"Loaded {len(documents)} documents, {len(all_chunks)} chunks")
for path, _ in documents:
    print(f"  - {path}")

Loaded 5 documents, 24 chunks
  - # Embeddings and Vector Stores

**Embeddings** are dense vector representations of text (or other data). Similar pieces of text get similar vectors, so we can find relevant content by comparing embeddings—e.g., with cosine similarity or Euclidean distance.

## How Embeddings Are Produced

Embedding models (e.g., OpenAI’s text-embedding-3, sentence-transformers like all-MiniLM-L6-v2) take text as input and output a fixed-size vector. They are trained so that semantically similar sentences have similar vectors. Dimensions are often 384, 768, or 1536, depending on the model.

## Vector Stores

A vector store is a database optimized for similarity search over embeddings. You insert (id, embedding, optional metadata and text), then query with an embedding and get the nearest neighbors. Examples include ChromaDB, Pinecone, Weaviate, and FAISS. ChromaDB is lightweight and can run locally, making it popular for prototyping and small-scale RAG.

## Use in RAG



## 3. Create embeddings and store in ChromaDB

In [3]:
# Use HuggingFace embeddings (no API key required)
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embeddings(texts: list):
    return embedding_model.encode(texts, show_progress_bar=len(texts) > 20).tolist()

In [4]:
texts = [c["content"] for c in all_chunks]
embeddings = get_embeddings(texts)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "AI articles chunks"}
)

ids = [f"chunk_{i}" for i in range(len(all_chunks))]
metadatas = [{"source": c["source"], "doc": c["doc"]} for c in all_chunks]
collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)

print(f"ChromaDB collection '{COLLECTION_NAME}' has {collection.count()} chunks.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ChromaDB collection 'ai_articles' has 24 chunks.


## 4. Visualize the embeddings

In [5]:
# Get all vectors and metadata from Chroma
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents_list = result["documents"]
metadatas_list = result["metadatas"]
doc_names = [m["doc"] for m in metadatas_list]

# Color by source document
unique_docs = sorted(set(doc_names))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
color_map = {d: colors[i % len(colors)] for i, d in enumerate(unique_docs)}
point_colors = [color_map[d] for d in doc_names]

In [ ]:
# Reduce to 2D with t-SNE for visualization
# Perplexity must be < n_samples (default 30 would fail with few chunks)
n_samples = len(vectors)
perplexity = min(30, max(5, n_samples - 1))
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
reduced = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced[:, 0],
    y=reduced[:, 1],
    mode="markers",
    marker=dict(size=6, color=point_colors, opacity=0.85),
    text=[f"Doc: {d}<br>{t[:120]}..." for d, t in zip(doc_names, documents_list)],
    hoverinfo="text"
)])
fig.update_layout(
    title="AI Articles — 2D embedding visualization (t-SNE)",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2",
    width=800,
    height=600,
    showlegend=False
)
fig.show()

## 5. Ask questions (retrieval + answer)

In [10]:
def retrieve(query: str, n: int = 4):
    """Return top-n chunks for the query."""
    q_emb = get_embeddings([query])[0]
    result = collection.query(
        query_embeddings=[q_emb],
        n_results=min(n, collection.count()),
        include=["documents", "metadatas", "distances"]
    )
    return result["documents"][0], result["metadatas"][0], result["distances"][0]

def ask(query: str, n_chunks: int = 4, use_llm: bool = True):
    """Retrieve relevant chunks and optionally generate an answer with OpenAI."""
    docs, metas, distances = retrieve(query, n=n_chunks)
    context = "\n\n---\n\n".join(docs) if docs else "(No relevant chunks found.)"
    
    print("Retrieved chunks:")
    for i, (doc, meta, d) in enumerate(zip(docs, metas, distances), 1):
        print(f"  [{i}] ({meta['doc']}) dist={d:.3f}\n      {doc[:100]}...")
    print()
    
    if use_llm:
        try:
            from openai import OpenAI
            
            client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "Answer based only on the context below. Be concise. If the context doesn't contain the answer, say so."},
                    {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
                ]
            )
            answer = resp.choices[0].message.content
            print("Answer:", answer)
            return answer
        except Exception as e:
            print("OpenAI not configured:", e)
            print("Showing retrieved context only.")
    
    print("Context (use_llm=False or no API key):\n", context[:1500], "...")
    return context

In [11]:
# Example questions — try your own!
ask("What is RAG and why is it used?", use_llm=True)

Retrieved chunks:
  [1] (embeddings_and_vector_stores) dist=1.029
      RAG

In RAG, document chunks are embedded and stored. At query time, the question is embedded and th...
  [2] (retrieval_augmented_generation) dist=1.157
      # Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation is a technique that combines ...
  [3] (large_language_models) dist=1.261
      gmented generation (RAG) combines LLMs with external knowledge bases to reduce hallucination and kee...
  [4] (retrieval_augmented_generation) dist=1.264
      RAG?

LLMs can hallucinate—generate plausible but incorrect information—and their knowledge is fixed...

Answer: RAG, or Retrieval-Augmented Generation, is a technique that combines a large language model (LLM) with an external knowledge base. It is used to reduce hallucination (generation of plausible but incorrect information) and to provide up-to-date answers by grounding responses in retrieved documents, allowing access to private or current data wi

'RAG, or Retrieval-Augmented Generation, is a technique that combines a large language model (LLM) with an external knowledge base. It is used to reduce hallucination (generation of plausible but incorrect information) and to provide up-to-date answers by grounding responses in retrieved documents, allowing access to private or current data without needing to retrain the model.'

In [12]:
ask("How do transformers use attention?", use_llm=True)

Retrieved chunks:
  [1] (transformers_and_attention) dist=0.652
      # Transformers and Attention

The transformer architecture, introduced in the 2017 paper "Attention ...
  [2] (transformers_and_attention) dist=0.823
      e step, unlike RNNs that process sequentially.

## Key Components

Transformers use **multi-head att...
  [3] (transformers_and_attention) dist=1.018
      ng each position.

## Self-Attention

In self-attention, each token is mapped to a query, key, and v...
  [4] (transformers_and_attention) dist=1.088
      en attention and feed-forward sub-layers, with residual connections and layer normalization for stab...

Answer: Transformers use attention through a mechanism called self-attention, where each token is mapped to a query, key, and value. The output at each position is a weighted sum of values, with weights determined by the similarity (e.g., dot product) between that position’s query and all keys. This allows the model to weigh the importance of every token 

'Transformers use attention through a mechanism called self-attention, where each token is mapped to a query, key, and value. The output at each position is a weighted sum of values, with weights determined by the similarity (e.g., dot product) between that position’s query and all keys. This allows the model to weigh the importance of every token in the sequence and capture long-range dependencies.'

In [13]:
# Without LLM: just see retrieved chunks
ask("What are embeddings and vector stores?", use_llm=False)

Retrieved chunks:
  [1] (embeddings_and_vector_stores) dist=0.800
      # Embeddings and Vector Stores

**Embeddings** are dense vector representations of text (or other da...
  [2] (embeddings_and_vector_stores) dist=0.857
      ’s text-embedding-3, sentence-transformers like all-MiniLM-L6-v2) take text as input and output a fi...
  [3] (retrieval_augmented_generation) dist=0.920
      1. **Indexing**: Documents are split into chunks, converted to vector embeddings, and stored in a ve...
  [4] (retrieval_augmented_generation) dist=1.207
      n**: The retrieved chunks are passed to the LLM as context, and the model generates an answer based ...

Context (use_llm=False or no API key):
 # Embeddings and Vector Stores

**Embeddings** are dense vector representations of text (or other data). Similar pieces of text get similar vectors, so we can find relevant content by comparing embeddings—e.g., with cosine similarity or Euclidean distance.

## How Embeddings Are Produced

Embedding model

'# Embeddings and Vector Stores\n\n**Embeddings** are dense vector representations of text (or other data). Similar pieces of text get similar vectors, so we can find relevant content by comparing embeddings—e.g., with cosine similarity or Euclidean distance.\n\n## How Embeddings Are Produced\n\nEmbedding models (e.g., OpenAI’s text-embedding-3, sentence-transformers like all-MiniLM-L6-v2) take text as i\n\n---\n\n’s text-embedding-3, sentence-transformers like all-MiniLM-L6-v2) take text as input and output a fixed-size vector. They are trained so that semantically similar sentences have similar vectors. Dimensions are often 384, 768, or 1536, depending on the model.\n\n## Vector Stores\n\nA vector store is a database optimized for similarity search over embeddings. You insert (id, embedding, optional metadata\n\n---\n\n1. **Indexing**: Documents are split into chunks, converted to vector embeddings, and stored in a vector database (e.g., ChromaDB, Pinecone).\n2. **Retrieval**: When t